In [ ]:
# src/ 側のコードを編集したら自動で再読み込みする
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from racing_prediction.data.scrape_shutuba import fetch_shutuba

# 出馬表から出走馬情報を取得 (User-Agent付き / アクセスごとに1~3秒ランダムsleep)
race_id = "202605050507"
df_shutuba = fetch_shutuba(race_id)
df_shutuba

In [ ]:
# 各馬の戦績ページから条件別 (距離・馬場状態・競馬場 × 芝/ダート) の平均着順を追加
# 対象レース当日以降の戦績は集計に含めない. 1頭ごとにアクセスするので時間がかかる
from racing_prediction.features.build_features import add_avg_rank_features

df_horses = add_avg_rank_features(df_shutuba)
df_horses

In [ ]:
# 各馬の血統ページから 父・母・父父・父母・母父・母母 (馬名とID) と父の系統を追加
from racing_prediction.data.scrape_horse_pedigree import add_pedigree

df_horses = add_pedigree(df_horses)
df_horses

In [ ]:
# 特徴量 + 目的変数 (is_top3) + 着順 のテーブルを作成し CSV に保存
# race_ids に複数のレースを入れると縦に結合して 1 つの CSV になる
from racing_prediction.data.make_dataset import make_dataset

race_ids = ["202605050507"]
output_path = Path.cwd().parent / "data" / "processed" / "dataset.csv"

df_dataset = make_dataset(race_ids, output_path)
df_dataset

In [ ]:
# 競馬場・期間・レース番号を指定して連続取得し, レースごとに CSV 保存
# 保存先: data/processed/races/{競馬場}/{開催日}/{レース番号}R.csv  (例: tokyo/20261122/07R.csv)
# 保存済みのレースは取得せずに読み込むので, 途中で止めても再実行すれば続きから取得できる
# src/ 側のコードを編集したら自動で再読み込みする
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from racing_prediction.data.make_dataset import collect_races

venue = "東京"                # 札幌・函館・福島・新潟・東京・中山・中京・京都・阪神・小倉
start_date = "2025-11-23"     # 集計期間 (開始日)
end_date = "2026-11-22"       # 集計期間 (終了日)
start_race_number = 7
end_race_number = 12
race_numbers = range(start_race_number, end_race_number + 1)   # 7R〜12R

df_races = collect_races(
    venue,
    start_date,
    end_date,
    race_numbers=race_numbers,
    output_dir=Path.cwd().parent / "data" / "processed" / "races",
)